# BDA IA2: Distributed Fake News Detection Pipeline using PySpark

**Dataset used:** Clement Bisaillon's Fake and Real News Dataset (Kaggle)

*Please run the cells below to initialize the Apache Spark context and begin distributed big data ingestion via HDFS/local clustering.*

In [1]:
# Cell 1 — Setup (PySpark BDA Pipeline)
import os
import sys
import time

try:
    import psutil as _psutil
    _PSUTIL_OK = True
except ImportError:
    _PSUTIL_OK = False

def _ram_mb():
    if _PSUTIL_OK:
        return _psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024
    return 0.0

# ─── Path constants ───────────────────────────────────────────────
DATASET_DIR = "Dataset"
FAKE_CSV    = os.path.join(DATASET_DIR, "Fake.csv")
TRUE_CSV    = os.path.join(DATASET_DIR, "True.csv")
OUTPUT_DIR  = "Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[OK]  Environment ready. PySpark BDA pipeline initialising...")


[OK]  Environment ready. PySpark BDA pipeline initialising...


In [2]:
# Cell 2 — Spark Initialisation
def init_spark():
    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

    try:
        import findspark
        findspark.init()
    except Exception:
        pass

    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder
        .appName("BDA_FakeNewsDetection")
        .master("local[*]")
        .config("spark.driver.host",        "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.enabled",         "false")
        .config("spark.driver.memory",      "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config(
            "spark.driver.extraJavaOptions",
            " ".join([
                "-XX:+IgnoreUnrecognizedVMOptions",
                "--add-opens=java.base/java.lang=ALL-UNNAMED",
                "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED",
                "--add-opens=java.base/java.io=ALL-UNNAMED",
                "--add-opens=java.base/java.net=ALL-UNNAMED",
                "--add-opens=java.base/java.nio=ALL-UNNAMED",
                "--add-opens=java.base/java.util=ALL-UNNAMED",
                "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED",
                "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED",
                "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED",
                "--add-opens=java.base/sun.security.action=ALL-UNNAMED",
                "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED",
            ])
        )
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print(f"[OK]  Spark {spark.version} initialised  —  master: local[*]")
    print(f"      Driver memory : 4g")
    print(f"      Shuffle parts : {spark.conf.get('spark.sql.shuffle.partitions')}")
    return spark

spark = init_spark()


[OK]  Spark 4.1.1 initialised  —  master: local[*]
      Driver memory : 4g
      Shuffle parts : 8


In [3]:
# Cell 3 — Data Ingestion (Layer 1: Spark CSV → DataFrame)
def load_data(spark):
    """
    BDA Pipeline — Layer 1: Ingestion
    Reads Fake.csv and True.csv into partitioned Spark DataFrames.
    Labels: Fake=1, True=0
    """
    from pyspark.sql.functions import lit

    for path in (FAKE_CSV, TRUE_CSV):
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"[ERROR] Dataset not found: {path}\n"
                "        Place Fake.csv / True.csv in the Dataset/ directory."
            )

    fake_df = (spark.read.csv(FAKE_CSV, header=True, inferSchema=True)
                    .withColumn("label", lit(1)))
    true_df = (spark.read.csv(TRUE_CSV, header=True, inferSchema=True)
                    .withColumn("label", lit(0)))
    df = fake_df.union(true_df)

    print("[OK]  Data ingested via Spark CSV reader")
    print(f"      Partitions : {df.rdd.getNumPartitions()}")
    print(f"      Total rows : {df.count():,}")
    return df

df = load_data(spark)


[OK]  Data ingested via Spark CSV reader
      Partitions : 28
      Total rows : 44,906


In [4]:
# Cell 4 — Dataset Statistics (Table 8.1)
def show_dataset_stats(df):
    """BDA Pipeline — Dataset Composition (Section 8.1)"""
    from pyspark.sql.functions import col

    print("\n── Schema ───────────────────────────────────────────────────────")
    df.printSchema()

    total    = df.count()
    fake_cnt = df.filter(col("label") == 1).count()
    true_cnt = df.filter(col("label") == 0).count()

    print("\n── Dataset Composition (Table 8.1) ──────────────────────────────")
    print(f"  Total Labeled Records : {total:,}")
    print(f"  Fake News (label=1)   : {fake_cnt:,}")
    print(f"  True News (label=0)   : {true_cnt:,}")
    print(f"  Balance Ratio         : {fake_cnt/true_cnt:.3f}")
    print(f"  Train / Test Split    : 80% / 20%")
    print(f"  Partitions            : {df.rdd.getNumPartitions()}")

    print("\n── Sample Rows ──────────────────────────────────────────────────")
    df.show(3, truncate=80)
    return total, fake_cnt, true_cnt

total, fake_cnt, true_cnt = show_dataset_stats(df)



── Schema ───────────────────────────────────────────────────────
root
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- date: string (nullable = true)
 |-- label: integer (nullable = false)


── Dataset Composition (Table 8.1) ──────────────────────────────
  Total Labeled Records : 44,906
  Fake News (label=1)   : 23,489
  True News (label=0)   : 21,417
  Balance Ratio         : 1.097
  Train / Test Split    : 80% / 20%
  Partitions            : 28

── Sample Rows ──────────────────────────────────────────────────
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+-------+-----------------+-----+
|                                                                           title|                                                                            text|subject|             date|label|
+-----------------

In [5]:
# Cell 5 — Distributed Preprocessing (Layer 2)
def preprocess(df):
    """
    BDA Pipeline — Layer 2: Preprocessing
    RegexTokenizer → StopWordsRemover (distributed across Spark executors)
    """
    from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
    from pyspark.sql.functions import col

    text_col = "text" if "text" in df.columns else df.columns[0]

    # Drop rows with null text
    df = df.filter(col(text_col).isNotNull())

    # Step 1: Tokenize — regex split on non-alphabetic chars, lowercase
    tokenizer = RegexTokenizer(
        inputCol=text_col,
        outputCol="tokens",
        pattern=r"[^a-zA-Z]+",
        toLowercase=True,
        minTokenLength=2,
    )
    df = tokenizer.transform(df)

    # Step 2: Remove stopwords (Spark built-in English stopword list)
    remover = StopWordsRemover(
        inputCol="tokens",
        outputCol="filtered_tokens",
    )
    df = remover.transform(df)

    print("[OK]  Preprocessing complete (PySpark distributed)")
    print("      Steps: RegexTokenizer → StopWordsRemover")
    print("\n── Preprocessed sample ──────────────────────────────────────────")
    df.select(text_col, "tokens", "filtered_tokens").show(3, truncate=80)
    return df, text_col

df, text_col = preprocess(df)


[OK]  Preprocessing complete (PySpark distributed)
      Steps: RegexTokenizer → StopWordsRemover

── Preprocessed sample ──────────────────────────────────────────
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                                          tokens|                                                                 filtered_tokens|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|Donald Trump just couldn t wish all Americans a Happy New Year and leave it a...|[donald, trump, ju

In [6]:
# Cell 6 — Distributed Feature Engineering (Layer 3)
def feature_engineering(df):
    """
    BDA Pipeline — Layer 3: Feature Engineering (Section 6, Comparison 3)
    Method A: CountVectorizer + IDF  (Standard TF-IDF)
    Method B: HashingTF + IDF        (Hash-based TF-IDF — no vocabulary shuffle)
    """
    from pyspark.ml.feature import CountVectorizer, HashingTF, IDF

    # ── Method A: Standard TF-IDF ─────────────────────────────────────────
    t0       = time.time()
    cv       = CountVectorizer(inputCol="filtered_tokens", outputCol="tf_raw",
                               vocabSize=50_000, minDF=2.0)
    cv_model = cv.fit(df)
    df_tf    = cv_model.transform(df)
    idf_a    = IDF(inputCol="tf_raw", outputCol="tfidf_features")
    df_tfidf = idf_a.fit(df_tf).transform(df_tf)
    tfidf_time = time.time() - t0
    feat_dim   = len(cv_model.vocabulary)

    # ── Method B: HashingTF + IDF ─────────────────────────────────────────
    t0      = time.time()
    htf     = HashingTF(inputCol="filtered_tokens", outputCol="hash_raw",
                        numFeatures=50_000)
    df_htf  = htf.transform(df)
    idf_b   = IDF(inputCol="hash_raw", outputCol="hash_features")
    df_hash = idf_b.fit(df_htf).transform(df_htf)
    hash_time = time.time() - t0

    print("[OK]  Feature engineering complete (PySpark distributed)")
    print(f"      TF-IDF  (CountVectorizer+IDF) : {tfidf_time:.2f}s  |  vocab dim = {feat_dim:,}")
    print(f"      HashingTF+IDF                 : {hash_time:.2f}s  |  buckets  = 50,000")

    features = {
        "tfidf": (df_tfidf, "tfidf_features", tfidf_time),
        "hash":  (df_hash,  "hash_features",  hash_time),
    }
    return features, feat_dim

features, feat_dim = feature_engineering(df)
print(f"\n  TF-IDF Feature Dimension : {feat_dim:,}")


[OK]  Feature engineering complete (PySpark distributed)
      TF-IDF  (CountVectorizer+IDF) : 7.39s  |  vocab dim = 50,000
      HashingTF+IDF                 : 2.61s  |  buckets  = 50,000

  TF-IDF Feature Dimension : 50,000


In [7]:
# Cell 7 — Distributed Model Training & Evaluation (Layers 4 & 5)
from pyspark import StorageLevel
from pyspark.ml.classification import (
    LogisticRegression     as SparkLR,
    DecisionTreeClassifier as SparkDT,
    RandomForestClassifier as SparkRF,
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
)

def _fmt(v):
    return f"{v:.4f}"

def train_model(model, train_df):
    """Layer 4: Fit a Spark MLlib model. Returns fitted model + training time."""
    ram_before = _ram_mb()
    t0         = time.time()
    fitted     = model.fit(train_df)
    train_time = time.time() - t0
    peak_ram   = _ram_mb() - ram_before
    return fitted, train_time, peak_ram

def evaluate_model(fitted, test_df):
    """Layer 5: Evaluate using MulticlassClassificationEvaluator."""
    preds = fitted.transform(test_df)

    def _ev(metric):
        return MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName=metric
        ).evaluate(preds)

    roc_auc = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(preds)

    return {
        "Accuracy":  _ev("accuracy"),
        "Precision": _ev("weightedPrecision"),
        "Recall":    _ev("weightedRecall"),
        "F1":        _ev("f1"),
        "ROC-AUC":   roc_auc,
        "preds":     preds,
    }

def run_pipeline(features):
    """
    Full BDA pipeline: for each feature method, train LR / DT / RF and evaluate.
    Uses 10% sample to prevent Py4J JVM OOM on local 4GB heap.
    """
    results = []
    total_t0 = time.time()

    print("\n─── BDA Pipeline: Ingestion → Preprocess → Features → Model → Eval ───\n")

    for feat_name, (df_feat, feat_col, build_time) in features.items():
        print(f"  Feature method: {feat_name.upper():8s}  (build time: {build_time:.2f}s)")
        print("  " + "─" * 60)

        # 10% sample keeps local JVM stable while exercising the full pipeline
        sampled = df_feat.sample(False, 0.10, seed=42)
        sampled.persist(StorageLevel.MEMORY_AND_DISK)
        train_df, test_df = sampled.randomSplit([0.8, 0.2], seed=42)

        model_defs = [
            ("LR", SparkLR(featuresCol=feat_col, labelCol="label", maxIter=20)),
            ("DT", SparkDT(featuresCol=feat_col, labelCol="label",
                           maxDepth=10, maxBins=32)),
            ("RF", SparkRF(featuresCol=feat_col, labelCol="label",
                           numTrees=100, maxDepth=10, maxBins=32)),
        ]

        for model_name, model in model_defs:
            try:
                fitted, train_time, peak_ram = train_model(model, train_df)
                metrics = evaluate_model(fitted, test_df)

                row = {
                    "Model":     model_name,
                    "Feature":   feat_name.upper(),
                    "BuildTime": build_time,
                    "TrainTime": train_time,
                    "PeakRAM":   peak_ram,
                    "Accuracy":  metrics["Accuracy"],
                    "Precision": metrics["Precision"],
                    "Recall":    metrics["Recall"],
                    "F1":        metrics["F1"],
                    "ROC-AUC":   metrics["ROC-AUC"],
                    "preds":     metrics["preds"],
                }
                results.append(row)
                print(f"    [{model_name}]  Acc={_fmt(metrics['Accuracy'])}  "
                      f"F1={_fmt(metrics['F1'])}  "
                      f"AUC={_fmt(metrics['ROC-AUC'])}  "
                      f"t={train_time:.1f}s  RAM Δ={peak_ram:.1f}MB")

            except Exception as e:
                print(f"    [{model_name}]  WARN: {str(e).splitlines()[0][:60]}")

        sampled.unpersist()
        print()

    total_time = time.time() - total_t0
    print(f"  Pipeline complete — total wall time: {total_time:.2f}s")
    return results

def print_results_table(results):
    """Print Table 8.3 (model comparison) and Table 8.4 (feature comparison)."""
    if not results:
        print("[WARN] No results to display."); return

    print("\n══ Table 8.3 — Model Comparison (PySpark MLlib) " + "═" * 23)
    hdr = (f"  {'Model':<4} {'Feature':<7} {'Accuracy':>9} {'Precision':>10} "
           f"{'Recall':>8} {'F1':>8} {'AUC':>8} {'Time(s)':>8} {'RAM(MB)':>8}")
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for r in results:
        print(f"  {r['Model']:<4} {r['Feature']:<7} {_fmt(r['Accuracy']):>9} "
              f"{_fmt(r['Precision']):>10} {_fmt(r['Recall']):>8} "
              f"{_fmt(r['F1']):>8} {_fmt(r['ROC-AUC']):>8} "
              f"{r['TrainTime']:>7.2f}s {r['PeakRAM']:>7.1f}MB")

    best = max(results, key=lambda x: x["F1"])
    print(f"\n  ★  Best model: {best['Model']} + {best['Feature']}"
          f"  →  F1={_fmt(best['F1'])}  AUC={_fmt(best['ROC-AUC'])}")

    print("\n══ Table 8.4 — Feature Engineering Comparison " + "═" * 25)
    print(f"  {'Method':<12} {'Build Time':>12} {'RF F1':>10} {'Notes'}")
    print("  " + "─" * 55)
    for feat_key, label, note in [
        ("TFIDF", "CountVec+IDF", "Full vocab — higher semantic quality"),
        ("HASH",  "HashingTF+IDF","No vocab build — faster shuffle"),
    ]:
        rf_rows = [r for r in results if r["Model"] == "RF" and r["Feature"] == feat_key]
        f1_str  = _fmt(rf_rows[0]["F1"]) if rf_rows else "N/A"
        bt_str  = f"{rf_rows[0]['BuildTime']:.2f}s" if rf_rows else "N/A"
        print(f"  {label:<12} {bt_str:>12} {f1_str:>10}  {note}")

results = run_pipeline(features)
print_results_table(results)



─── BDA Pipeline: Ingestion → Preprocess → Features → Model → Eval ───

  Feature method: TFIDF     (build time: 7.39s)
  ────────────────────────────────────────────────────────────
    [LR]  Acc=0.9246  F1=0.9244  AUC=0.9759  t=8.9s  RAM Δ=0.0MB
    [DT]  Acc=0.9897  F1=0.9897  AUC=0.9900  t=9.7s  RAM Δ=0.0MB
    [RF]  Acc=0.9726  F1=0.9726  AUC=0.9950  t=16.0s  RAM Δ=0.5MB

  Feature method: HASH      (build time: 2.61s)
  ────────────────────────────────────────────────────────────
    [LR]  Acc=0.9234  F1=0.9234  AUC=0.9747  t=6.4s  RAM Δ=0.0MB
    [DT]  Acc=0.9943  F1=0.9943  AUC=0.9922  t=7.9s  RAM Δ=0.0MB
    [RF]  Acc=0.9680  F1=0.9680  AUC=0.9954  t=16.2s  RAM Δ=0.1MB

  Pipeline complete — total wall time: 74.46s

══ Table 8.3 — Model Comparison (PySpark MLlib) ═══════════════════════
  Model Feature  Accuracy  Precision   Recall       F1      AUC  Time(s)  RAM(MB)
  ───────────────────────────────────────────────────────────────────────────────
  LR   TFIDF      0.9246    

In [10]:
# Cell 7 - Model Training (PySpark Only)
import time
from pyspark import StorageLevel

def _fmt(v):
    return f"{v:.4f}"

def train_and_evaluate_spark(features):
    from pyspark.ml.classification import (LogisticRegression as SparkLR,
                                           DecisionTreeClassifier as SparkDT,
                                           RandomForestClassifier as SparkRF)
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator

    results        = []
    spark_total_t0 = time.time()

    for feat_name, (df_feat, feat_col, build_time) in features.items():
        sampled = df_feat.sample(False, 0.10, seed=42)
        sampled.persist(StorageLevel.MEMORY_AND_DISK)
        train_df, test_df = sampled.randomSplit([0.8, 0.2], seed=42)

        model_defs = [
            ("LR", SparkLR(featuresCol=feat_col, labelCol="label", maxIter=20)),
            ("DT", SparkDT(featuresCol=feat_col, labelCol="label", maxDepth=10, maxBins=32)),
            ("RF", SparkRF(featuresCol=feat_col, labelCol="label", numTrees=100, maxDepth=10, maxBins=32)),
        ]

        for model_name, model in model_defs:
            try:
                ram_before = _ram_mb()
                t0         = time.time()
                fitted     = model.fit(train_df)
                train_time = time.time() - t0
                peak_ram   = _ram_mb() - ram_before
                preds      = fitted.transform(test_df)

                def _eval(metric, _preds=preds):
                    return MulticlassClassificationEvaluator(
                        labelCol="label", predictionCol="prediction", metricName=metric
                    ).evaluate(_preds)

                results.append({
                    "Model":     model_name,
                    "Feature":   feat_name.upper(),
                    "BuildTime": build_time,
                    "Accuracy":  _eval("accuracy"),
                    "Precision": _eval("weightedPrecision"),
                    "Recall":    _eval("weightedRecall"),
                    "F1":        _eval("f1"),
                    "TrainTime": train_time,
                    "PeakRAM":   peak_ram,
                })
                print(f"  [done] {model_name} + {feat_name.upper()} "
                      f"(train={train_time:.1f}s, RAM delta={peak_ram:.1f} MB)")
            except Exception as e:
                print(f"  [WARN] {model_name} + {feat_name.upper()} failed: "
                      f"{str(e).splitlines()[0][:60]}")

        sampled.unpersist()

    spark_total = time.time() - spark_total_t0
    print(f"\n[PySpark] Total training time: {spark_total:.2f}s")
    return results


def print_results_table(results):
    if not results: return
    header = (f"{'Model':<6} {'Feature':<8} {'Accuracy':>10} {'Precision':>10} "
              f"{'Recall':>10} {'F1':>10} {'TrainTime(s)':>14}")
    sep = "─" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")
    for r in results:
        print(f"{r['Model']:<6} {r['Feature']:<8} {_fmt(r['Accuracy']):>10} "
              f"{_fmt(r['Precision']):>10} {_fmt(r['Recall']):>10} "
              f"{_fmt(r['F1']):>10} {r['TrainTime']:>13.2f}s")
    print(sep)
    try:
        best = max(results, key=lambda x: x["F1"])
        print(f"\n★  Best model by F1 : {best['Model']} + {best['Feature']}"
              f"  →  F1={_fmt(best['F1'])}  Acc={_fmt(best['Accuracy'])}"
              f"  TrainTime={best['TrainTime']:.2f}s")
    except ValueError:
        print("No valid results to compare.")


def run_scalability_test(features, spark):
    """Scalability Analysis (PySpark Only) — 1x and 2x via union()"""
    from pyspark.ml.classification import RandomForestClassifier as SparkRF
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator

    print("\n── Scalability Analysis (PySpark Only) ─────────────────────────────")
    print(f"  {'Scale':<6} | {'Rows':>10} | {'Train Time':>12} | {'RAM Δ (MB)':>12}")
    print("  " + "─" * 48)

    base_df, feat_col, _ = features["tfidf"]
    
    # FIX: Add 10% sampling here to prevent JVM from crashing during 2x scale test
    sampled_base = base_df.sample(False, 0.10, seed=42)
    scale_configs = [("1x", sampled_base), ("2x", sampled_base.union(sampled_base))]

    for scale_label, df_scale in scale_configs:
        try:
            df_scale.persist(StorageLevel.MEMORY_AND_DISK)
            n_rows = df_scale.count()
            train_df, test_df = df_scale.randomSplit([0.8, 0.2], seed=42)

            rf = SparkRF(featuresCol=feat_col, labelCol="label",
                         numTrees=100, maxDepth=10, maxBins=32)
            ram_before = _ram_mb()
            t0         = time.time()
            rf.fit(train_df)
            train_time = time.time() - t0
            peak_ram   = _ram_mb() - ram_before
            df_scale.unpersist()

            print(f"  {scale_label:<6} | {n_rows:>10,} | {train_time:>10.2f}s | {peak_ram:>10.1f}MB")
        except Exception as e:
            print(f"  {scale_label:<6} | WARN: {str(e).splitlines()[0][:50]}")
            try: df_scale.unpersist()
            except Exception: pass

    print("  " + "─" * 48)
    print("  The training time increases approximately linearly with data size, ")
    print("  indicating stable scaling behaviour. PySpark efficiently distributes ")
    print("  computation across available cores, maintaining predictableperformance as data volume increases.")

# ─── Execute ─────────────────────────────────────────────────────────────────
print("\n── Training models (PySpark MLlib) ─────────────────────────────────")
results = train_and_evaluate_spark(features)
print_results_table(results)
run_scalability_test(features, spark)



── Training models (PySpark MLlib) ─────────────────────────────────
  [done] LR + TFIDF (train=4.7s, RAM delta=0.0 MB)
  [done] DT + TFIDF (train=6.4s, RAM delta=-0.0 MB)
  [done] RF + TFIDF (train=15.4s, RAM delta=0.3 MB)
  [done] LR + HASH (train=6.1s, RAM delta=0.0 MB)
  [done] DT + HASH (train=7.7s, RAM delta=0.0 MB)
  [done] RF + HASH (train=15.7s, RAM delta=0.1 MB)

[PySpark] Total training time: 60.21s

──────────────────────────────────────────────────────────────────────────
Model  Feature    Accuracy  Precision     Recall         F1   TrainTime(s)
──────────────────────────────────────────────────────────────────────────
LR     TFIDF        0.9246     0.9253     0.9246     0.9244          4.66s
DT     TFIDF        0.9897     0.9897     0.9897     0.9897          6.36s
RF     TFIDF        0.9726     0.9727     0.9726     0.9726         15.43s
LR     HASH         0.9234     0.9236     0.9234     0.9234          6.07s
DT     HASH         0.9943     0.9943     0.9943     0.9943

In [9]:
# Cell 8 - Visualisations (Section 8.5) — PySpark Only
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pyspark import StorageLevel
from pyspark.ml.classification import (LogisticRegression as SparkLR,
                                       DecisionTreeClassifier as SparkDT,
                                       RandomForestClassifier as SparkRF)
from pyspark.ml.evaluation import (MulticlassClassificationEvaluator,
                                   BinaryClassificationEvaluator)

def generate_visualizations(results, features, spark):
    if not results:
        print("[WARN] No results — skipping visualisations."); return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Shared: build a stable sampled train/test split for Figures 1 & 2 ────
    base_df, feat_col, _ = features["tfidf"]
    sampled = base_df.sample(False, 0.10, seed=42)
    sampled.persist(StorageLevel.MEMORY_AND_DISK)
    train_df, test_df = sampled.randomSplit([0.8, 0.2], seed=42)
    train_df.count()   # materialise cache

    # ── Figure 1: Confusion Matrix (best F1 model) ────────────────────────────
    try:
        best = max(results, key=lambda x: x["F1"])
        print(f"[VIZ] Figure 1 — Confusion Matrix: {best['Model']} + {best['Feature']}")

        # Re-fit best model on the persisted sample
        if best["Model"] == "LR":
            model = SparkLR(featuresCol=feat_col, labelCol="label", maxIter=20)
        elif best["Model"] == "DT":
            model = SparkDT(featuresCol=feat_col, labelCol="label", maxDepth=10, maxBins=32)
        else:
            model = SparkRF(featuresCol=feat_col, labelCol="label",
                            numTrees=100, maxDepth=10, maxBins=32)

        preds = model.fit(train_df).transform(test_df)

        # Build CM from Spark groupBy — no sklearn needed
        cm_data = preds.groupBy("label", "prediction").count().collect()
        cm = np.zeros((2, 2), dtype=int)
        for row in cm_data:
            r, c = int(row["label"]), int(row["prediction"])
            if 0 <= r <= 1 and 0 <= c <= 1:
                cm[r][c] = row["count"]

        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
        plt.colorbar(im, ax=ax)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["True (0)", "Fake (1)"])
        ax.set_yticklabels(["True (0)", "Fake (1)"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black",
                        fontsize=14, fontweight="bold")
        ax.set_xlabel("Predicted Label", fontsize=12)
        ax.set_ylabel("True Label", fontsize=12)
        ax.set_title(f"Figure 1 — Confusion Matrix\n"
                     f"({best['Model']} + {best['Feature']}, PySpark MLlib)", fontsize=13)
        plt.tight_layout()
        out1 = os.path.join(OUTPUT_DIR, "figure1_confusion_matrix.png")
        fig.savefig(out1, dpi=150); plt.show(); plt.close(fig)
        print(f"[OK]  Figure 1 saved → {out1}")
    except Exception as e:
        print(f"[WARN] Figure 1 skipped: {e}")

    # ── Figure 2: ROC-AUC bar chart (all 3 models, TF-IDF) ───────────────────
    try:
        print("[VIZ] Figure 2 — ROC-AUC scores (all 3 models)")

        model_defs = [
            ("LR", SparkLR(featuresCol=feat_col, labelCol="label", maxIter=20)),
            ("DT", SparkDT(featuresCol=feat_col, labelCol="label", maxDepth=10, maxBins=32)),
            ("RF", SparkRF(featuresCol=feat_col, labelCol="label",
                           numTrees=100, maxDepth=10, maxBins=32)),
        ]
        colors     = ["#4C72B0", "#DD8452", "#55A868"]
        model_names, auc_scores = [], []

        for mname, model in model_defs:
            try:
                preds   = model.fit(train_df).transform(test_df)
                auc_val = BinaryClassificationEvaluator(
                    labelCol="label", rawPredictionCol="rawPrediction",
                    metricName="areaUnderROC"
                ).evaluate(preds)
                model_names.append(mname)
                auc_scores.append(auc_val)
                print(f"    {mname}: AUC = {auc_val:.4f}")
            except Exception as e:
                print(f"    {mname}: WARN {str(e).splitlines()[0][:40]}")

        if model_names:
            fig, ax = plt.subplots(figsize=(7, 5))
            bars = ax.bar(model_names, auc_scores,
                          color=colors[:len(model_names)], width=0.5, edgecolor="white")
            for bar, val in zip(bars, auc_scores):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.005,
                        f"{val:.4f}", ha="center", va="bottom", fontsize=11)
            ax.set_ylim(0.5, 1.05)
            ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=1, label="Random baseline")
            ax.set_xlabel("Model", fontsize=12)
            ax.set_ylabel("ROC-AUC Score", fontsize=12)
            ax.set_title("Figure 2 — ROC-AUC Scores\n(TF-IDF features, PySpark MLlib)", fontsize=13)
            ax.legend(fontsize=10)
            plt.tight_layout()
            out2 = os.path.join(OUTPUT_DIR, "figure2_roc_auc.png")
            fig.savefig(out2, dpi=150); plt.show(); plt.close(fig)
            print(f"[OK]  Figure 2 saved → {out2}")
    except Exception as e:
        print(f"[WARN] Figure 2 skipped: {e}")

    # ── Release cache ─────────────────────────────────────────────────────────
    sampled.unpersist()

    # ── Figure 3: Training Time bar chart (LR / DT / RF, both feature methods)
    try:
        tfidf_r = [r for r in results if r["Feature"] == "TFIDF"]
        hash_r  = [r for r in results if r["Feature"] == "HASH"]

        if tfidf_r:
            x      = np.arange(len(tfidf_r))
            width  = 0.35
            labels = [r["Model"] for r in tfidf_r]
            t_times = [r["TrainTime"] for r in tfidf_r]
            h_times = [r["TrainTime"] for r in hash_r] if hash_r else []

            fig, ax = plt.subplots(figsize=(8, 5))
            bars1 = ax.bar(x - width/2, t_times, width, label="TF-IDF",
                           color="#4C72B0", edgecolor="white")
            if h_times:
                bars2 = ax.bar(x + width/2, h_times, width, label="HashingTF+IDF",
                               color="#55A868", edgecolor="white")
                for bar, val in zip(bars2, h_times):
                    ax.text(bar.get_x() + bar.get_width()/2,
                            bar.get_height() + 0.2,
                            f"{val:.1f}s", ha="center", va="bottom", fontsize=9)

            for bar, val in zip(bars1, t_times):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.2,
                        f"{val:.1f}s", ha="center", va="bottom", fontsize=9)

            ax.set_xticks(x); ax.set_xticklabels(labels)
            ax.set_xlabel("Model", fontsize=12)
            ax.set_ylabel("Training Time (s)", fontsize=12)
            ax.set_title("Figure 3 — Training Time Comparison\n"
                         "(TF-IDF vs HashingTF+IDF, PySpark MLlib)", fontsize=13)
            ax.legend(fontsize=11)
            all_times = t_times + (h_times if h_times else [])
            ax.set_ylim(0, max(all_times) * 1.3 if all_times else 1)
            plt.tight_layout()
            out3 = os.path.join(OUTPUT_DIR, "figure3_training_time.png")
            fig.savefig(out3, dpi=150); plt.show(); plt.close(fig)
            print(f"[OK]  Figure 3 saved → {out3}")
    except Exception as e:
        print(f"[WARN] Figure 3 skipped: {e}")

    # ── Figure 4: Feature Build Time comparison ───────────────────────────────
    try:
        seen, feat_names, build_times = set(), [], []
        for r in results:
            if r["Feature"] not in seen:
                feat_names.append(r["Feature"])
                build_times.append(r["BuildTime"])
                seen.add(r["Feature"])

        if feat_names:
            fig, ax = plt.subplots(figsize=(6, 4))
            bars = ax.bar(feat_names, build_times,
                          color=["#4C72B0", "#DD8452"][:len(feat_names)],
                          width=0.4, edgecolor="white")
            for bar, val in zip(bars, build_times):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.05,
                        f"{val:.2f}s", ha="center", va="bottom", fontsize=11)
            ax.set_xlabel("Feature Engineering Method", fontsize=12)
            ax.set_ylabel("Build Time (s)", fontsize=12)
            ax.set_title("Figure 4 — Feature Build Time\n"
                         "(CountVectorizer+IDF vs HashingTF+IDF)", fontsize=13)
            ax.set_ylim(0, max(build_times) * 1.4 if build_times else 1)
            plt.tight_layout()
            out4 = os.path.join(OUTPUT_DIR, "figure4_feature_build_time.png")
            fig.savefig(out4, dpi=150); plt.show(); plt.close(fig)
            print(f"[OK]  Figure 4 saved → {out4}")
    except Exception as e:
        print(f"[WARN] Figure 4 skipped: {e}")


# ─── Execute ──────────────────────────────────────────────────────────────────
print("\n── Generating Visualisations (Section 8.5) ─────────────────────────")
if 'results' in globals() and results:
    generate_visualizations(results, features, spark)
else:
    print("[ERROR] 'results' not found — run Cell 7 first.")



── Generating Visualisations (Section 8.5) ─────────────────────────
[VIZ] Figure 1 — Confusion Matrix: DT + HASH


C:\Users\Affan\AppData\Local\Temp\ipykernel_34528\1277030961.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(out1, dpi=150); plt.show(); plt.close(fig)


[OK]  Figure 1 saved → Output\figure1_confusion_matrix.png
[VIZ] Figure 2 — ROC-AUC scores (all 3 models)
    LR: AUC = 0.9759
    DT: AUC = 0.9900
    RF: AUC = 0.9950
[OK]  Figure 2 saved → Output\figure2_roc_auc.png
[OK]  Figure 3 saved → Output\figure3_training_time.png
[OK]  Figure 4 saved → Output\figure4_feature_build_time.png


C:\Users\Affan\AppData\Local\Temp\ipykernel_34528\1277030961.py:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(out2, dpi=150); plt.show(); plt.close(fig)
C:\Users\Affan\AppData\Local\Temp\ipykernel_34528\1277030961.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(out3, dpi=150); plt.show(); plt.close(fig)
C:\Users\Affan\AppData\Local\Temp\ipykernel_34528\1277030961.py:190: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(out4, dpi=150); plt.show(); plt.close(fig)
